# Step 3 — classify station objects: real railway stations vs. urban transit

Reads the filtered station extract from step 2 (`data/step2_output_eu_stations.osm.pbf`) and produces the `station_mode` classification described in `README.md`'s "Design background" §3 Stage A. See `README.md` for where this sits in the overall pipeline.

**Inputs**
- `data/step2_output_eu_stations.osm.pbf` — every OSM station object in Europe, with all tags (step 2 output)
- `data/step3a_output_way_relation_centers.csv` — center coordinates for station ways/relations (step 3a output)

**Output**
- `data/step3b_output_osm_stations_classified.csv` — one row per station object, with `station_mode`/`mode_rule` columns. Nothing is dropped; downstream steps filter on `station_mode`.

**Why the centers CSV is needed:** the filtered extract contains only the matched objects themselves. Station *nodes* carry their own coordinates, but station *ways* and *relations* only reference member node IDs that are not in the file — without step 3a their coordinates cannot be resolved, and ~14% of all station objects (13,500 ways + 553 relations, including many large stations mapped as areas) would silently fall out of the pipeline.

In [ ]:
from pathlib import Path

import osmium
import pandas as pd

STATION_EXTRACT = Path("data/step2_output_eu_stations.osm.pbf")  # step 2 output
CENTERS_CSV = Path("data/step3a_output_way_relation_centers.csv")  # step 3a output
OUTPUT_CSV = Path("data/step3b_output_osm_stations_classified.csv")

for required in (STATION_EXTRACT, CENTERS_CSV):
    if not required.exists():
        raise FileNotFoundError(
            f"{required} is missing — run the earlier pipeline step that produces it (see README.md)."
        )

## 1. Extract station objects

Stations can be OSM **nodes** (a single point — the common case, ~86%), **ways** (an area, e.g. a station building outline), or **relations** (grouped station parts). Nodes provide their own coordinates; ways and relations get theirs from the step 3a centers lookup. We only need a representative point for the later geo-matching, not the station footprint.

Only the tags the Stage A rules actually need are pulled out explicitly; the full raw tag dict is kept too in case a rule needs tuning later.

In [ ]:
centers_df = pd.read_csv(CENTERS_CSV)
centers = {
    (row.osm_type, row.osm_id): (row.lat, row.lon)
    for row in centers_df.itertuples()
}
print(f"{len(centers):,} way/relation centers loaded")


class StationHandler(osmium.SimpleHandler):
    """Collects station nodes, ways, and relations with their classification-relevant tags."""

    STATION_RAILWAY_VALUES = {"station", "halt"}

    def __init__(self):
        super().__init__()
        self.rows: list[dict] = []
        self.unresolved: list[tuple[str, int]] = []  # ways/relations missing from the centers CSV

    def _collect(self, osm_id: int, osm_type: str, lat: float, lon: float, tags) -> None:
        tags = dict(tags)
        railway = tags.get("railway")
        public_transport = tags.get("public_transport")

        # Defensive re-check even though step 2 already filtered on exactly these tags
        if railway not in self.STATION_RAILWAY_VALUES and public_transport != "station":
            return

        self.rows.append(
            {
                "osm_type": osm_type,
                "osm_id": osm_id,
                "name": tags.get("name"),
                "lat": lat,
                "lon": lon,
                "country": tags.get("addr:country"),
                "railway": railway,
                "public_transport": public_transport,
                "uic_ref": tags.get("uic_ref"),
                "train": tags.get("train"),
                "station": tags.get("station"),
                "subway": tags.get("subway"),
                "tram": tags.get("tram"),
                "light_rail": tags.get("light_rail"),
                "amenity": tags.get("amenity"),
                "all_tags": tags,
            }
        )

    def _collect_via_center(self, osm_id: int, osm_type: str, tags) -> None:
        center = centers.get((osm_type, osm_id))
        if center is None:
            self.unresolved.append((osm_type, osm_id))
            return
        self._collect(osm_id, osm_type, center[0], center[1], tags)

    def node(self, n) -> None:
        if n.location.valid():
            self._collect(n.id, "node", n.location.lat, n.location.lon, n.tags)

    def way(self, w) -> None:
        self._collect_via_center(w.id, "way", w.tags)

    def relation(self, r) -> None:
        self._collect_via_center(r.id, "relation", r.tags)


handler = StationHandler()
handler.apply_file(str(STATION_EXTRACT))

osm = pd.DataFrame(handler.rows)
print(f"{len(osm):,} station objects collected")
print(osm["osm_type"].value_counts())
if handler.unresolved:
    print(
        f"WARNING: {len(handler.unresolved):,} ways/relations have no center in {CENTERS_CSV} "
        "and were skipped — re-run step 3a if this number is more than a small handful."
    )

## 2. Stage A — classify heavy rail vs. urban transit

Direct implementation of `README.md`'s "Design background" §3 Stage A. Read the two helper functions together with the doc's rule list — the code mirrors it rule-for-rule:

- `train=yes` or `uic_ref` present → heavy-rail evidence.
- `station=subway`/`light_rail`, or `subway=yes`/`tram=yes`/`light_rail=yes` → transit evidence.
- Both present → `mixed` (keep — big hubs serve rail and metro under one OSM object).
- Only heavy-rail evidence → `heavy_rail`.
- Only transit evidence → `urban_transit` (drop from further evaluation).
- `amenity=ferry_terminal` without heavy-rail evidence → `other` with `mode_rule=ferry_terminal` (ferry piers carry `public_transport=station` surprisingly often — 6,134 objects in the Europe extract — and would otherwise flood the undecided bucket).
- Neither, but `station=` has some other, unrecognized value (funicular, ferry, monorail, ...) → `other` — mark it, don't guess.
- Neither, and no `station=` tag at all → `undecided` (no transit indicators present).

One easy-to-miss pitfall: `bool(float("nan"))` is `True` in Python, so a naive `bool(row["uic_ref"])` on a pandas column silently treats *every* row missing that tag as if it had one. All presence checks below go through `pd.isna`/`pd.notna` explicitly.

In [ ]:
def _split_tag_values(value) -> set[str]:
    """OSM tags can hold semicolon-separated multi-values; a missing tag (NaN) becomes an empty set."""
    if pd.isna(value):
        return set()
    return {v.strip() for v in str(value).lower().split(";") if v.strip()}


KNOWN_TRANSIT_STATION_VALUES = {"subway", "light_rail", "tram"}


def classify_station_mode(row: pd.Series) -> tuple[str, str]:
    """Return (station_mode, mode_rule) per README.md's "Design background" Stage A."""
    station_tags = _split_tag_values(row["station"])
    is_known_transit = (
        bool(station_tags & KNOWN_TRANSIT_STATION_VALUES)
        or row["subway"] == "yes"
        or row["tram"] == "yes"
        or row["light_rail"] == "yes"
    )
    has_uic_ref = pd.notna(row["uic_ref"]) and str(row["uic_ref"]).strip() != ""
    has_heavy_rail_evidence = row["train"] == "yes" or has_uic_ref

    if has_heavy_rail_evidence and is_known_transit:
        return "mixed", "heavy_rail_evidence_plus_transit_tag"
    if has_heavy_rail_evidence:
        return "heavy_rail", "train_yes_or_uic_ref"
    if is_known_transit:
        return "urban_transit", "transit_tag_without_heavy_rail_evidence"
    if row["amenity"] == "ferry_terminal":
        return "other", "ferry_terminal"
    if station_tags - KNOWN_TRANSIT_STATION_VALUES:
        return "other", "unrecognized_station_value"
    return "undecided", "no_mode_indicators_present"

In [ ]:
osm[["station_mode", "mode_rule"]] = osm.apply(lambda r: pd.Series(classify_station_mode(r)), axis=1)
osm["station_mode"].value_counts()

### Sanity check: undecided bucket size

Per the design doc, a large `undecided` bucket is a signal to add a track-proximity pass before trusting it. With the full Europe extract this bucket is known to be **large (roughly 40%)** — mostly small halts tagged with nothing but `railway=halt`. They stay in per the keep-it-in principle; the track-proximity refinement is an open item in `README.md`.

In [ ]:
undecided_share = (osm["station_mode"] == "undecided").mean()
print(f"{undecided_share:.1%} of station objects are undecided")
osm.loc[osm["station_mode"] == "undecided", ["osm_id", "name", "station"]].head(20)

### Sanity check: country coverage

`addr:country` is tagged on well under 1% of station objects, so the `country` column is nearly empty here. That is expected at this stage — **filling it from coordinates is a prerequisite for step 4** (matching is done within-country). Use the project's existing `CountryIndex` helper for that rather than adding a new reverse-geocoding dependency; see the open items in `README.md`.

In [ ]:
country_coverage = osm["country"].notna().mean()
print(f"{country_coverage:.1%} of station objects carry addr:country")

## 3. Output — GTFS-style stops, ready for the next steps

`urban_transit` rows are kept (not dropped) per the non-destructive principle — later steps filter on `station_mode` as needed, but nothing is deleted here. The `stop_id` is the OSM id, type-prefixed (`osm:n…`/`osm:w…`/`osm:r…`) so it's recognizable as OSM-sourced and won't collide with IDs from other sources later.

In [ ]:
result = osm[
    ["osm_id", "name", "lat", "lon", "country", "station_mode", "mode_rule", "uic_ref", "osm_type"]
].rename(
    columns={
        "osm_id": "stop_id",
        "name": "stop_name",
        "lat": "stop_lat",
        "lon": "stop_lon",
        "uic_ref": "stop_code",
    }
)
result["stop_id"] = "osm:" + result["osm_type"].str[0] + result["stop_id"].astype(str)  # e.g. osm:n123456
result = result.drop(columns="osm_type")

result.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(result)} rows to {OUTPUT_CSV}")
result.head()